# Phase 3: LLM Integration
This notebook demonstrates how to connect our existing retrieval system to a Large Language Model (LLM).

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Ollama

print("LangChain modules loaded.")

In [ ]:
from src.core.config import settings
# 1. Initialize Vector Store
embedding_function = HuggingFaceEmbeddings(model_name=settings.EMBEDDING_MODEL_NAME)
vectorstore = Chroma(
    collection_name=settings.COLLECTION_NAME,
    embedding_function=embedding_function,
    persist_directory=settings.CHROMA_DB_PATH
)

# 2. Initialize LLM (Local Ollama in this case)
llm = Ollama(model=settings.OLLAMA_MODEL)
print("System loaded!")

### Step 1: Query the vector store

In [ ]:
query = "What is the main topic of the document?"
results = vectorstore.similarity_search_with_score(query, k=3)

context_pieces = []
for doc, score in results:
    context_pieces.append(doc.page_content)

context_str = "\n\n".join(context_pieces)
print("Retrieved Context:\n", context_str[:200], "...")

### Step 2: Generate Answer

In [ ]:
template = """You are a helpful AI assistant. Use the following pieces of retrieved context to answer the user's question. 
If you don't know the answer based on the context, just say that you don't know. Do not make up information.

--- CONTEXT ---
{context}
---------------

User Question: {query}
Answer:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "query"]
)

# Create the chain and execute it
chain = prompt | llm
answer = chain.invoke({
    "context": context_str,
    "query": query
})

print("Answer:\n", answer)